# EdgeForge AI on a free Colab GPU

One coarse layout in, a labelled computer-vision dataset out. This notebook runs the real engine (the FastAPI app in `src/app/main.py`) on Colab's free T4 and calls it the way the web UI does.

**Before you run it:** *Runtime → Change runtime type* → **T4 GPU**, and **Runtime version 2026.07** (Python 3.12). The pinned `numpy<2` has no wheels for Python 3.13, which newer runtimes use.

The first run downloads about 10 GB of model weights (SDXL base, the SDXL Canny ControlNet, the SDXL VAE and YOLOv8n), which takes a few minutes. Each 1024×1024 image then takes about a minute on a T4.

In [ ]:
import sys
assert sys.version_info < (3, 13), "Pick Runtime > Change runtime type > Runtime version 2026.07 (Python 3.12)."
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
!git clone -q https://github.com/yashsinha1702/edgeforge-ai.git /content/edgeforge-ai
!pip -q install -r /content/edgeforge-ai/requirements.txt "transformers==4.44.2" "accelerate==0.34.2"
# diffusers 0.25 imports peft when it is installed, and Colab's peft needs a newer huggingface_hub
!pip -q uninstall -y peft

## Start the engine

The same command as the Dockerfile, in the background. Loading the models takes about a minute once they are downloaded; the cell waits for `Factory Ready!` in the log.

In [ ]:
!cd /content/edgeforge-ai && nohup env PYTHONPATH=src python -m uvicorn src.app.main:app --host 127.0.0.1 --port 8000 > /content/engine.log 2>&1 &
!timeout 1800 bash -c 'until grep -q "Factory Ready" /content/engine.log; do if grep -q "Traceback" /content/engine.log; then cat /content/engine.log; exit 1; fi; sleep 5; done' && tail -3 /content/engine.log

## Generate a small dataset

`docs/layout.png` is a coarse white car on black, the kind of input the README describes. The batch endpoint remixes the layout for every image (1–3 cars, shifted, scaled and flipped), samples a condition and a seed, generates, and labels the result with YOLOv8.

In [ ]:
!curl -s -X POST http://127.0.0.1:8000/generate_batch \
  -F "intent=a car on a highway" -F "batch_size=4" \
  -F "control_image=@/content/edgeforge-ai/docs/layout.png" \
  -o /content/edgeforge_dataset.zip && unzip -o -q /content/edgeforge_dataset.zip -d /content/dataset && ls -R /content/dataset | head -20

In [ ]:
import json
from pathlib import Path
from PIL import Image, ImageDraw
from IPython.display import display

COCO = {2: "car", 3: "motorcycle", 5: "bus", 7: "truck", 0: "person"}
root = Path("/content/dataset")
prompts = {p["image"]: p for p in json.loads((root / "prompts.json").read_text())}
tiles = []
for img_path in sorted((root / "images").glob("*.png")):
    im = Image.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(im)
    for line in (root / "labels" / (img_path.stem + ".txt")).read_text().split("\n"):
        if not line.strip():
            continue
        cls, x, y, w, h = line.split()
        x, y, w, h = (float(v) * im.width for v in (x, y, w, h))
        draw.rectangle([x - w / 2, y - h / 2, x + w / 2, y + h / 2], outline="#ffd400", width=5)
        draw.text((x - w / 2 + 6, y - h / 2 + 4), COCO.get(int(cls), cls), fill="#ffd400")
    print(img_path.name, "|", prompts[f"images/{img_path.name}"]["condition"])
    tiles.append(im.resize((512, 512)))
grid = Image.new("RGB", (512 * len(tiles), 512))
for i, t in enumerate(tiles):
    grid.paste(t, (512 * i, 0))
display(grid)

The zip is a YOLO-format dataset: `images/`, `labels/` (one `class x_center y_center width height` line per box, normalised) and `prompts.json`, which records the prompt, condition and seed behind each image. The class IDs are COCO indices from the pretrained YOLOv8n, so spot-check the labels before you train on them.